In [ ]:
import os
import sys
# 假设 dataset0 文件夹在当前工作目录下
module_path = os.path.abspath(os.path.join('./Agrobot Dataset/dataset0'))
if module_path not in sys.path:
    sys.path.append(module_path)
print(os.getcwd()) 

In [ ]:
import tensorflow as tf

# 检查 TensorFlow 版本
print("TensorFlow version:", tf.__version__)

# 检查可用的 GPU 数量
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# 配置 GPU 按需分配内存
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
from gtda.time_series import SlidingWindow
import matplotlib.pyplot as plt
from math import atan2, pi, sqrt, cos, sin, floor
import tensorflow as tf
from tensorflow.python.keras.backend import set_session
config = tf.compat.v1.ConfigProto() 
config.gpu_options.allow_growth = True  
config.log_device_placement = True  
sess2 = tf.compat.v1.Session(config=config)
set_session(sess2)  
from tensorflow.keras.layers import Dense, MaxPooling1D, Flatten, Reshape
from tensorflow.keras import Input, Model
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow.keras.backend as K
from tensorflow.keras.models import load_model
from tcn import TCN, tcn_full_summary
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform
import pickle
import csv
import random
import itertools
import math
import time
from data_utils_0 import *
from traj_utils import *
from torchviz import make_dot
import tensorboard

## Import Dataset

In [ ]:
# 创建函数以计算 FLOPS
def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(tf.TensorSpec([1] + model.input_shape[1:], model.inputs[0].dtype))

    frozen_func, graph_def = convert_variables_to_constants_v2_as_graph(concrete_func)

    with tf.Graph().as_default() as graph:
        tf.import_graph_def(graph_def, name='')
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd='op', options=opts)
        return flops.total_float_ops

In [ ]:
window_size = 100
stride = 20

f = './Agrobot Dataset/dataset0/'
X_train,Y_Pos_train, Physics_Vec_train, x_vel_train, y_vel_train, x0_list_train, y0_list_train, size_of_each_train = import_agrobot_dataset_p1(dataset_folder=f, type_flag=1, window_size=window_size, stride=stride)
P = np.repeat(Physics_Vec_train,window_size).reshape((Physics_Vec_train.shape[0],window_size,1)) #得到训练数据集
X_train = np.concatenate((X_train,P),axis=2) #按列拼接得到第十个频域特征

X_test,Y_Pos_test, Physics_Vec_test, x_vel_test, y_vel_test, x0_list_test, y0_list_test, size_of_each_test= import_agrobot_dataset_p1(type_flag = 2, dataset_folder=f,window_size=window_size, stride=stride)
P_test = np.repeat(Physics_Vec_test,window_size).reshape((Physics_Vec_test.shape[0],window_size,1)) #得到测试数据集
X_test = np.concatenate((X_test,P_test),axis=2)

## Model Training

In [ ]:
from tensorflow.keras.utils import plot_model
nb_filters = 32
kernel_size = 5
dilations = [1,2,4,8,16,32,64,128]
dropout_rate = 0.0
use_skip_connections = True

batch_size, timesteps, input_dim = 256, window_size, X_train.shape[2]
i = Input(shape=(timesteps, input_dim))

m = TCN(nb_filters=nb_filters,kernel_size=kernel_size,dilations=dilations,dropout_rate=dropout_rate,
            use_skip_connections=use_skip_connections)(i)  

# 打印 TCN 层的输出形状
print("TCN output shape:", m.shape)

# 使用 Reshape 调整形状
m = Reshape((nb_filters, 1))(m)  # 调整为 (batch_size, nb_filters, 1)

m = MaxPooling1D(pool_size=(2))(m)
m = Flatten()(m)
m = Dense(32, activation='linear', name='pre')(m)
output1 = Dense(1, activation='linear', name='velx')(m)
output2 = Dense(1, activation='linear', name='vely')(m)
model = Model(inputs=[i], outputs=[output1, output2])
opt = tf.keras.optimizers.Adam()
model.compile(loss={'velx': 'mse','vely':'mse'},optimizer=opt)  
model.summary()

# 可视化模型
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

In [ ]:
model_name = 'Agrobot_First_TCN.keras'
checkpoint = ModelCheckpoint(model_name, monitor='loss', verbose=1, save_best_only=True)
model.fit(x=X_train, y=[x_vel_train, y_vel_train],epochs=3000, shuffle=True,callbacks=[checkpoint],batch_size=batch_size)     #训练模型

## Evaluation

In [ ]:
#model_name = 'Agrobot_First_TCN.keras'
model.save('./Neural Kalman IMU GNSS Fusion/PreTrained Models/Agrobot_First_TCN.keras')

Unseen Trajectories

In [ ]:
#测试部分提取轨迹与计算ATE，RTE
ATE = []
RTE = []
ATE_dist = []
RTE_dist = []
for i in range(len(size_of_each_test)):
    Pvx, Pvy = model_pos_generator(X_test, size_of_each_test, 
                   x0_list_test, y0_list_test, window_size, stride,i,model)   #i指文件序号 返回轨迹数据
    Gvx, Gvy = GT_pos_generator(x_vel_test,y_vel_test,size_of_each_test, 
                                x0_list_test, y0_list_test, window_size, stride,i)  #返回轨迹真值数据
    
    at, rt, at_all, rt_all = Cal_TE(Gvx, Gvy, Pvx, Pvy,
                                    sampling_rate=100,window_size=window_size,stride=stride) #返回ATE,RTE，轨迹中所有点的偏差，一分钟内轨迹的点的偏差
    ATE.append(at)
    RTE.append(rt)
    ATE_dist.append(Cal_len_meters(Gvx, Gvy)) #计算轨迹的总距离
    RTE_dist.append(Cal_len_meters(Gvx, Gvy, 600)) #计算一分钟内轨迹的总距离
    print('ATE, RTE, Trajectory Length, Trajectory Length (60 seconds)',ATE[i],RTE[i],ATE_dist[i],RTE_dist[i])
    
print('Median ATE and RTE', np.median(ATE),np.median(RTE))
print('Mean ATE and RTE', np.mean(ATE),np.mean(RTE))
print('STD ATE and RTE', np.std(ATE),np.std(RTE))

Seen Trajectories

In [ ]:
#训练部分提取轨迹与计算ATE,RTE
ATE = []
RTE = []
ATE_dist = []
RTE_dist = []
for i in range(len(size_of_each_train)):
    Pvx, Pvy = model_pos_generator(X_train, size_of_each_train, 
                   x0_list_train, y0_list_train, window_size, stride,i,model)   
    Gvx, Gvy = GT_pos_generator(x_vel_train,y_vel_train,size_of_each_train,
                                x0_list_train, y0_list_train, window_size, stride,i)
    
    at, rt, at_all, rt_all = Cal_TE(Gvx, Gvy, Pvx, Pvy,
                                    sampling_rate=100,window_size=window_size,stride=stride)
    ATE.append(at)
    RTE.append(rt)
    ATE_dist.append(Cal_len_meters(Gvx, Gvy))
    RTE_dist.append(Cal_len_meters(Gvx, Gvy, 600))
    print('ATE, RTE, Trajectory Length, Trajectory Length (60 seconds)',ATE[i],RTE[i],ATE_dist[i],RTE_dist[i])
    
print('Median ATE and RTE', np.median(ATE),np.median(RTE))
print('Mean ATE and RTE', np.mean(ATE),np.mean(RTE))
print('STD ATE and RTE', np.std(ATE),np.std(RTE))

Plot

In [ ]:
Pvx, Pvy = model_pos_generator(X_train, size_of_each_train, 
               x0_list_train, y0_list_train, window_size, stride,0,model)   #第一个数据集的预测轨迹
Gvx, Gvy = GT_pos_generator(x_vel_train,y_vel_train,size_of_each_train,
                            x0_list_train, y0_list_train, window_size, stride,0) #第一个数据集的真实轨迹 可以自己调节看轨迹的预测结果与真实数据的对比

In [ ]:
plt.plot(Gvx[0:2000],Gvy[0:2000])
plt.plot(Pvx[0:2000],Pvy[0:2000])

In [ ]:
print(Cal_len_meters(Gvx[0:2250],Gvy[0:2250]))